In [8]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
BFND = f"{DRIVE}/BFND-data"
CODE_ZIP = f"{DRIVE}/BFND_step5_code.zip"
SHEET = f"{BFND}/data/annotations/evidence_eval_sheet.csv"
META = f"{BFND}/features/meta.csv"
EVIDENCE = f"{BFND}/data/evidence/evidence.csv"
LABELLED = f"{BFND}/data/annotations/dev_new_passages_labelled.csv"

for path, what in [(CODE_ZIP, "BFND_step5_code.zip"), (SHEET, "evidence_eval_sheet.csv"),
                   (META, "features/meta.csv"), (EVIDENCE, "data/evidence/evidence.csv"),
                   (LABELLED, "dev_new_passages_labelled.csv")]:
    assert os.path.exists(path), f"Missing {what}. Expected at: {path}"
print("Drive mounted. All five input files found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. All five input files found.


In [9]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0",
                "rank-bm25>=0.2.2", "pytest"], check=True)
import sentence_transformers, rank_bm25
print("sentence-transformers", sentence_transformers.__version__, "| rank-bm25 ready")

sentence-transformers 5.7.0 | rank-bm25 ready


In [10]:
import os, zipfile, subprocess, sys

EXTRACT = "/content/bfnd_step6"
os.makedirs(EXTRACT, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(EXTRACT)

PROJECT = None
for root, dirs, files in os.walk(EXTRACT):
    if "src" in dirs and "scripts" in dirs and "configs" in dirs:
        PROJECT = root
        break
assert PROJECT is not None, "src/, scripts/ and configs/ not found in the zip"
for f in ["scripts/14_dev_retriever_overlap.py", "scripts/15_dev_retriever_compare.py"]:
    assert os.path.exists(os.path.join(PROJECT, f)), f"{f} is missing from the zip. Re-upload the new zip."
print("Project folder:", PROJECT)

Project folder: /content/bfnd_step6/BFND_code


In [11]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q",
                    "tests/test_evidence_eval.py", "tests/test_evidence_app.py", "tests/test_ocr_metrics.py"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1500:])
r.check_returncode()
print("Tests passed")

...................................                                      [100%]
35 passed in 1.12s

Tests passed


In [12]:
import pandas as pd

sheet = pd.read_csv(SHEET, encoding="utf-8-sig", dtype=str, keep_default_na=False)
dev = sheet[sheet["role"] == "dev"]
eval_rows = sheet[sheet["role"] == "eval"]
assert (dev["relevant_0_or_1"].str.strip() != "").all(), "Some dev rows in the sheet are not labelled."
assert (eval_rows["relevant_0_or_1"].str.strip() == "").all(), "Eval rows have labels. Stop."
print("Sheet check passed: dev all labelled, eval all blank.")

lab = pd.read_csv(LABELLED, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print("New labelled rows:", len(lab))
assert (lab["relevant_0_or_1"].str.strip().isin(["0", "1"])).all(), "New labelled file has blank or other labels."
print("New labelled file check passed:", lab["relevant_0_or_1"].value_counts().to_dict())

Sheet check passed: dev all labelled, eval all blank.
New labelled rows: 71
New labelled file check passed: {'0': 65, '1': 6}


In [13]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/14_dev_retriever_overlap.py"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("---- stderr ----")
    print(r.stderr[-4000:])
r.check_returncode()

{
  "posts": 40,
  "hybrid_pairs": 120,
  "hybrid_pairs_already_labelled": 49,
  "unlabelled_hybrid_pairs": 71,
  "posts_with_new_pair": 39,
  "alpha": 0.5,
  "top_k": 3,
  "embed_model": "intfloat/multilingual-e5-base",
  "device": "cuda",
  "note": "Pair-level: a (post, passage) pair counts as labelled only if that post was labelled with that passage. Passages are matched by their first 400 characters. Dev labels only."
}
New (post, passage) pairs to label: 71 (see /content/drive/MyDrive/BFND-data/results/evidence/dev_new_passages.csv)



In [14]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/15_dev_retriever_compare.py", "--new-labelled", LABELLED],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("---- stderr ----")
    print(r.stderr[-4000:])
r.check_returncode()

Wrote /content/drive/MyDrive/BFND-data/results/evidence/dev_retriever_compare.json
BM25   share 0.242  p@1 0.325  any 0.5
Hybrid share 0.225  p@1 0.35  any 0.475



In [15]:
import json
res = json.load(open(f"{BFND}/results/evidence/dev_retriever_compare.json", encoding="utf-8"))
print(json.dumps(res, ensure_ascii=False, indent=2))

{
  "posts": 40,
  "bm25_top3_pairs": 120,
  "hybrid_top3_pairs": 120,
  "hybrid_pairs_from_sheet": 49,
  "hybrid_pairs_from_new_labels": 71,
  "bm25": {
    "share_all_3": {
      "value": 0.242,
      "ci95": [
        0.158,
        0.325
      ]
    },
    "precision_at_1": {
      "value": 0.325,
      "ci95": [
        0.2,
        0.475
      ]
    },
    "any_relevant_top3": {
      "value": 0.5,
      "ci95": [
        0.35,
        0.65
      ]
    },
    "relevant_by_rank": {
      "1": 13,
      "2": 11,
      "3": 5
    },
    "posts_with_no_relevant": 20
  },
  "hybrid": {
    "share_all_3": {
      "value": 0.225,
      "ci95": [
        0.142,
        0.317
      ]
    },
    "precision_at_1": {
      "value": 0.35,
      "ci95": [
        0.2,
        0.5
      ]
    },
    "any_relevant_top3": {
      "value": 0.475,
      "ci95": [
        0.325,
        0.625
      ]
    },
    "relevant_by_rank": {
      "1": 14,
      "2": 7,
      "3": 6
    },
    "posts_with_no

In [16]:
import time, shutil
stamp = time.strftime("%Y%m%d_%H%M")
EXPORT = f"{BFND}/exports/step6_{stamp}"
os.makedirs(EXPORT, exist_ok=True)
shutil.copytree(f"{BFND}/results/evidence", f"{EXPORT}/evidence", dirs_exist_ok=True)
print("Exported to:", f"{EXPORT}/evidence")

Exported to: /content/drive/MyDrive/BFND-data/exports/step6_20261127_1521/evidence
